# Practice Lab: Function Calling & Tool Use (Lesson 7.5)

Module 7 · 8 exercises · Tool schemas + ReAct + MCP + India Stack

Exercises 1, 2, 3, 4, 6, 7 run locally.


# Lesson 7.5: Function Calling & Tool Use

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 3, 4, 6, 7 run **locally**. Ex 5, 8 are architecture/design.


In [ ]:
import json, re
from enum import IntEnum


---
## Exercise 1 (Easy): Multi-Provider Tool Calling


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

def get_weather(city: str) -> dict:
    data = {"hyderabad": {"temp": 34, "condition": "Sunny"},
            "mumbai": {"temp": 31, "condition": "Humid"}}
    return data.get(city.lower(), {"error": f"No data: {city}"})

def search_courses(topic: str, max_price: float = 50000) -> list:
    courses = [{"name": "GenAI", "price": 29999, "topic": "generative ai"},
               {"name": "Python", "price": 9999, "topic": "python"},
               {"name": "ML Eng", "price": 39999, "topic": "machine learning"}]
    return [c for c in courses if topic.lower() in c["topic"] and c["price"] <= max_price]

def calculate_emi(principal: float, rate: float, months: int) -> dict:
    r = rate / 100 / 12
    emi = principal * r * (1+r)**months / ((1+r)**months - 1)
    return {"emi": round(emi, 2), "total": round(emi*months, 2)}

class ToolOrchestrator:
    def __init__(self):
        self.tools = {}
    def register(self, name, func, desc, params):
        self.tools[name] = {"func": func, "desc": desc, "params": params}
    def execute(self, name, args):
        if name not in self.tools:
            return {"error": f"Unknown: {name}", "ok": False}
        try:
            return {"result": self.tools[name]["func"](**args), "ok": True}
        except Exception as e:
            return {"error": str(e), "ok": False}
    def to_openai(self):
        return [{"type": "function", "function": {"name": n, "description": t["desc"],
                 "parameters": t["params"]}} for n, t in self.tools.items()]
    def to_anthropic(self):
        return [{"name": n, "description": t["desc"], "input_schema": t["params"]}
                for n, t in self.tools.items()]
    def to_gemini(self):
        return [t["func"] for t in self.tools.values()]

orch = ToolOrchestrator()
ws = {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
cs = {"type": "object", "properties": {"topic": {"type": "string"}, "max_price": {"type": "number"}}, "required": ["topic"]}
es = {"type": "object", "properties": {"principal": {"type": "number"}, "rate": {"type": "number"}, "months": {"type": "integer"}}, "required": ["principal", "rate", "months"]}

orch.register("get_weather", get_weather, "Get weather", ws)
orch.register("search_courses", search_courses, "Search courses", cs)
orch.register("calculate_emi", calculate_emi, "Calculate EMI", es)

print(f"OpenAI: {len(orch.to_openai())} tools")
print(json.dumps(orch.to_openai()[0], indent=2)[:150])
print(f"\nAnthropic: {len(orch.to_anthropic())} tools")
print(f"Gemini: {len(orch.to_gemini())} Python functions")

print("\nExecution:")
print(f"  Weather: {orch.execute('get_weather', {'city': 'mumbai'})}")
print(f"  Courses: {orch.execute('search_courses', {'topic': 'python'})}")
print(f"  EMI: {orch.execute('calculate_emi', {'principal': 30000, 'rate': 12, 'months': 6})}")
print(f"  Unknown: {orch.execute('hack', {})}")

print("\nResult format: OpenAI=role:tool, Anthropic=tool_result in user, Gemini=FunctionResponse")


</details>


---
## Exercise 2 (Easy): tool_choice Forcing


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
modes = {
    "auto": {"openai": '"auto"', "anthropic": '{"type":"auto"}', "gemini": "AUTO",
             "when": "Conversational - model decides freely"},
    "required": {"openai": '"required"', "anthropic": '{"type":"any"}', "gemini": "ANY",
                 "when": "Agent loops - must act every turn"},
    "specific": {"openai": '{"type":"function","function":{"name":"X"}}',
                 "anthropic": '{"type":"tool","name":"X"}',
                 "gemini": 'ANY+allowedFunctionNames',
                 "when": "Deterministic extraction pipeline"},
    "none": {"openai": '"none"', "anthropic": '{"type":"none"}', "gemini": "NONE",
             "when": "Force text-only (last iteration)"},
}

print(f"{'Mode':<12} {'OpenAI':<35} {'Anthropic':<22} {'Gemini'}")
print("=" * 75)
for mode, info in modes.items():
    print(f"{mode:<12} {info['openai']:<35} {info['anthropic']:<22} {info['gemini']}")

print("\nUse Cases:")
for mode, info in modes.items():
    print(f"  {mode.upper()}: {info['when']}")

print("\nFake Tool Pattern (Anthropic):")
print("  Define tool with desired JSON schema as input_schema")
print("  Force with tool_choice: {type: 'tool', name: 'extract'}")
print("  Result in: response.content[0].input (structured JSON)")

print("\nExtended Thinking Limitation:")
print("  Forced tool_choice INCOMPATIBLE with thinking=True")
print("  Only 'auto' and 'none' work with thinking enabled")


</details>


---
## Exercise 3 (Medium): ReAct Agent Loop


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

def search(query):
    db = {"RAG": "Retrieval-Augmented Generation combines search with LLMs",
          "embeddings": "Dense vector representations, 768-1536 dims",
          "Hyderabad weather": "34C, Sunny, 45% humidity"}
    for k, v in db.items():
        if k.lower() in query.lower(): return v
    return f"No results: {query}"

def calculator(expr):
    try: return str(eval(expr, {"__builtins__": {}}))
    except: return f"Error: {expr}"

def get_time(tz="IST"):
    return f"2026-05-12 14:30 {tz}"

TOOLS = {"search": search, "calculator": calculator, "get_time": get_time}

class ReactAgent:
    def __init__(self, tools, max_iters=5, max_tokens=500, max_errors=3):
        self.tools = tools
        self.max_iters = max_iters
        self.max_tokens = max_tokens
        self.max_errors = max_errors
        self.tokens = 0
        self.errors = 0
        self.iters = 0
        self.last_calls = []

    def _sim_llm(self, msgs):
        last_user = next((m["content"] for m in reversed(msgs) if m["role"] == "user"), "")
        if any(m.get("role") == "tool_result" for m in msgs):
            tr = next(m["content"] for m in reversed(msgs) if m.get("role") == "tool_result")
            return {"type": "text", "content": f"Based on results: {tr[:80]}"}
        q = last_user.lower()
        if "weather" in q: return {"type": "tool", "name": "search", "args": {"query": "Hyderabad weather"}}
        if "time" in q: return {"type": "tool", "name": "get_time", "args": {"tz": "IST"}}
        if "rag" in q: return {"type": "tool", "name": "search", "args": {"query": "RAG"}}
        return {"type": "text", "content": f"Answer: {q}"}

    def run(self, query):
        msgs = [{"role": "user", "content": query}]
        self.tokens = len(query.split()) * 13 // 10
        for i in range(self.max_iters):
            self.iters = i + 1
            if self.tokens > self.max_tokens:
                return f"[STOP] Token budget ({self.tokens}/{self.max_tokens})"
            if self.errors >= self.max_errors:
                return f"[STOP] Error threshold ({self.errors})"
            resp = self._sim_llm(msgs)
            if resp["type"] == "text":
                print(f"  [{i+1}] DONE: {resp['content'][:60]}")
                return resp["content"]
            sig = f"{resp['name']}({json.dumps(resp['args'])})"
            if sig in self.last_calls[-3:]:
                return f"[STOP] No progress: repeated {resp['name']}"
            self.last_calls.append(sig)
            print(f"  [{i+1}] ACTION: {resp['name']}({resp['args']})")
            try:
                result = self.tools[resp["name"]](**resp["args"])
                self.errors = 0
                print(f"       RESULT: {str(result)[:60]}")
                msgs.append({"role": "tool_result", "content": str(result)})
                self.tokens += len(str(result).split()) * 13 // 10
            except Exception as e:
                self.errors += 1
                msgs.append({"role": "tool_result", "content": f"Error: {e}"})
        return "[STOP] Max iterations"

for q in ["What's the weather?", "What is RAG?", "What time is it?"]:
    agent = ReactAgent(TOOLS)
    print(f"\nQuery: {q}")
    agent.run(q)
    print(f"  Iters: {agent.iters}, Tokens: {agent.tokens}")

print("\n5 Termination Conditions:")
for i, c in enumerate(["Natural completion", "Max iterations", "Token budget",
                        "No-progress detection", "Error threshold"], 1):
    print(f"  {i}. {c}")


</details>


---
## Exercise 4 (Medium): Streaming Tool Calls


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

class ToolCallAccumulator:
    def __init__(self):
        self.calls = {}
    def process(self, delta):
        if "tool_calls" not in delta: return
        for tc in delta["tool_calls"]:
            idx = tc["index"]
            if idx not in self.calls:
                self.calls[idx] = {"name": "", "args": ""}
            fn = tc.get("function", {})
            if "name" in fn: self.calls[idx]["name"] = fn["name"]
            if "arguments" in fn: self.calls[idx]["args"] += fn["arguments"]
    def get_parsed(self):
        results = []
        for idx in sorted(self.calls):
            c = self.calls[idx]
            try:
                results.append({"name": c["name"], "args": json.loads(c["args"]), "valid": True})
            except:
                results.append({"name": c["name"], "args": c["args"], "valid": False})
        return results

deltas = [
    {"tool_calls": [{"index": 0, "function": {"name": "get_weather"}}]},
    {"tool_calls": [{"index": 0, "function": {"arguments": '{"ci'}}]},
    {"tool_calls": [{"index": 1, "function": {"name": "search_courses"}}]},
    {"tool_calls": [{"index": 0, "function": {"arguments": 'ty":"Mumbai"}'}}]},
    {"tool_calls": [{"index": 1, "function": {"arguments": '{"topic":"ai","max_price":35000}'}}]},
]

acc = ToolCallAccumulator()
print("Streaming Tool Calls (parallel):")
for i, d in enumerate(deltas):
    acc.process(d)
    for idx in sorted(acc.calls):
        c = acc.calls[idx]
        print(f"  [{i+1}] call[{idx}] {c['name']:<16} {c['args']}")

results = acc.get_parsed()
print(f"\nParsed ({len(results)}):")
for r in results:
    print(f"  {r['name']}({r['args']}) [{'OK' if r['valid'] else 'BAD'}]")

print(f"\nAll valid: {all(r['valid'] for r in results)}")
print(f"Rule: NEVER parse until finish_reason='tool_calls'")


</details>


---
## Exercise 6 (Challenge): Human-in-the-Loop Agent


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
from enum import IntEnum

class Tier(IntEnum):
    READ = 1; WRITE = 2; ADMIN = 3

class SecureOrch:
    def __init__(self, role="user"):
        self.tools = {}
        self.role = role
        self.blocked = []
    def register(self, name, func, tier, desc=""):
        self.tools[name] = {"func": func, "tier": tier, "desc": desc}
    def execute(self, name, args, approved=False):
        if name not in self.tools:
            return {"error": f"Unknown: {name}", "ok": False}
        t = self.tools[name]
        if t["tier"] == Tier.READ:
            return {"result": t["func"](**args), "ok": True, "tier": "READ"}
        if t["tier"] == Tier.WRITE:
            if not approved:
                return {"needs_approval": True, "ok": False, "tier": "WRITE"}
            return {"result": t["func"](**args), "ok": True, "tier": "WRITE"}
        if t["tier"] == Tier.ADMIN:
            if self.role != "admin":
                self.blocked.append(name)
                return {"error": "Admin required", "ok": False, "blocked": True}
            if not approved:
                return {"needs_approval": True, "ok": False, "tier": "ADMIN"}
            return {"result": t["func"](**args), "ok": True, "tier": "ADMIN"}

orch = SecureOrch(role="user")
orch.register("search", lambda q: f"Results: {q}", Tier.READ)
orch.register("get_weather", lambda city: f"{city}: 34C", Tier.READ)
orch.register("enroll", lambda c, s: f"Enrolled {s} in {c}", Tier.WRITE)
orch.register("send_email", lambda to, subj: f"Email to {to}", Tier.WRITE)
orch.register("delete_account", lambda uid: f"Deleted {uid}", Tier.ADMIN)
orch.register("refund", lambda oid, amt: f"Refund {amt}", Tier.ADMIN)

tests = [
    ("READ auto", "search", {"q": "RAG"}, False),
    ("WRITE no approval", "enroll", {"c": "GenAI", "s": "Rahul"}, False),
    ("WRITE approved", "enroll", {"c": "GenAI", "s": "Rahul"}, True),
    ("ADMIN (user role)", "delete_account", {"uid": "123"}, False),
    ("ADMIN (user+approved)", "delete_account", {"uid": "123"}, True),
]

print("Permission Tier Tests:")
print("=" * 55)
for label, tool, args, approved in tests:
    r = orch.execute(tool, args, approved=approved)
    status = "OK" if r.get("ok") else "PENDING" if r.get("needs_approval") else "BLOCKED"
    print(f"  [{status:>7}] {label}")

print(f"\nBlocked attempts: {len(orch.blocked)}")
print(f"Framework hooks > prompt instructions")


</details>


---
## Exercise 7 (Challenge): India Stack Agent


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re

def validate_gstin(gstin):
    pattern = r'^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1}$'
    valid = bool(re.match(pattern, gstin))
    if not valid: return {"valid": False, "error": "Invalid format"}
    states = {"27": "Maharashtra", "36": "Telangana", "29": "Karnataka"}
    return {"valid": True, "state": states.get(gstin[:2], "Other"), "pan": gstin[2:12]}

def upi_payment(vpa, amount, purpose):
    if not re.match(r'^[\w.-]+@[\w]+$', vpa): return {"error": "Invalid VPA"}
    if amount > 100000: return {"error": "Exceeds UPI limit (1L)"}
    return {"status": "pending_approval", "vpa": vpa, "amount": amount,
            "requires_2fa": True, "message": "Requires user approval + UPI PIN"}

def verify_aadhaar(last4, name):
    if len(last4) != 4 or not last4.isdigit(): return {"error": "Need exactly 4 digits"}
    return {"verified": True, "masked": f"XXXX XXXX {last4}",
            "pii": "Full Aadhaar never stored"}

print("India Stack Tools:")
print("=" * 55)

print("\n1. GST Validation:")
for g in ["27AABCU9603R1ZM", "36AADCN1234F1ZP", "INVALID"]:
    r = validate_gstin(g)
    print(f"  {'OK' if r.get('valid') else 'BAD':>3} {g} -> {r.get('state', r.get('error'))}")

print("\n2. UPI Payments:")
for vpa, amt in [("shop@upi", 4999), ("invalid", 500), ("x@paytm", 200000)]:
    r = upi_payment(vpa, amt, "Course")
    print(f"  {vpa:<15} Rs {amt:>8} -> {r.get('status', r.get('error'))}")

print("\n3. Aadhaar (PII-safe):")
for l4, name in [("9012", "Rahul"), ("123", "Bad"), ("5678", "Priya")]:
    r = verify_aadhaar(l4, name)
    print(f"  {l4:<6} -> {r.get('masked', r.get('error'))}")

print("\nPII Rules:")
for r in ["NEVER store full Aadhaar", "NEVER log in observability",
          "Only last 4 in API params", "DPDPA: explicit consent", "Rs 250 Cr penalty"]:
    print(f"  ! {r}")


</details>


---
## Exercise 5 (Medium): MCP Server
Architecture/design. See practice-lab-7_5.html.


In [ ]:
# MCP Server Architecture (design; runnable, prints the FastMCP pattern)
mcp_design = {
    "server_name": "netsetos-tools",
    "transport": "Streamable HTTP (remote) or stdio (local)",
    "tools": [
        {"name": "search_courses", "desc": "Search Netsetos courses",
         "params": ["topic: str", "max_price: float"]},
        {"name": "get_student_progress", "desc": "Get student progress",
         "params": ["student_id: str"]},
        {"name": "calculate_emi", "desc": "Calculate course EMI",
         "params": ["principal: float", "rate: float", "months: int"]},
    ],
    "resources": [
        {"uri": "netsetos://courses", "desc": "Course catalog"},
        {"uri": "netsetos://pricing", "desc": "Current pricing"},
    ],
}

print("MCP Server Design:")
print("=" * 55)
print(f"Server: {mcp_design['server_name']}")
print(f"Transport: {mcp_design['transport']}")

print(f"\nTools ({len(mcp_design['tools'])}):")
for t in mcp_design["tools"]:
    print(f"  @mcp.tool()")
    print(f"  def {t['name']}({', '.join(t['params'])})")
    print(f"    # {t['desc']}")

print(f"\nResources ({len(mcp_design['resources'])}):")
for r in mcp_design["resources"]:
    print(f"  {r['uri']}: {r['desc']}")

# FastMCP pattern
print("\n\nFastMCP Code Pattern:")
print("""  from fastmcp import FastMCP
  
  mcp = FastMCP("netsetos-tools")
  
  @mcp.tool()
  def search_courses(topic: str, max_price: float = 50000):
      # Auto-generates JSON Schema from type hints!
      ...
  
  @mcp.resource("netsetos://courses")
  def get_courses():
      return course_catalog_json
  
  mcp.run(transport="streamable-http", port=8000)""")

# MCP lifecycle
print("\nMCP Lifecycle:")
lifecycle = [
    "initialize (capabilities exchange)",
    "tools/list (discover available tools)",
    "tools/call (execute a tool)",
    "resources/read (fetch data)",
    "notifications (progress updates)",
    "shutdown (cleanup)",
]
for i, step in enumerate(lifecycle):
    print(f"  {i+1}. {step}")

# Security concerns
print("\nMCP Security Concerns:")
concerns = [
    "43% of early servers vulnerable to prompt injection",
    "Tool poisoning: malicious tool descriptions",
    "Cross-server shadowing: tool name conflicts",
    "Credential exposure in tool results",
    "No built-in auth (OAuth 2.1 added in spec v2)",
    '"The S in MCP stands for security" (community joke)',
]
for c in concerns:
    print(f"  ! {c}")

---
## Exercise 8 (Challenge): Production Agent
Architecture/design. See practice-lab-7_5.html.


In [ ]:
# Production Agent Architecture
architecture = {
    "components": [
        {"name": "ReAct Loop", "role": "Agent control flow",
         "config": "max_iters=15, token_budget=50K"},
        {"name": "ToolOrchestrator", "role": "Unified tool registry",
         "config": "3-tier permissions, error handling"},
        {"name": "Rate Limiter", "role": "Per-conversation limits",
         "config": "20 tool calls/conv, 100K tokens/hr"},
        {"name": "Input Validator", "role": "Pydantic schemas",
         "config": "Validate all tool arguments before execution"},
        {"name": "Sandbox", "role": "Execution isolation",
         "config": "Docker --cap-drop ALL (minimum)"},
        {"name": "Langfuse", "role": "Observability",
         "config": "Cost tracking, trace-level attribution"},
    ],
    "deployment": {
        "platform": "Cloud Run (asia-south1)",
        "state": "Redis (sessions + rate limits)",
        "secrets": "Secret Manager",
        "monitoring": "Langfuse + structlog",
    },
    "safety": [
        "Never eval() or exec() LLM output",
        "Parameterized queries (never string concat)",
        "Tool argument validation (Pydantic)",
        "Permission tiers (framework hooks)",
        "Rate limiting (per-user, per-conversation)",
        "Token budget enforcement",
        "OWASP LLM Top 10 compliance",
    ],
}

print("Production Agent Architecture:")
print("=" * 55)
print(f"\nComponents ({len(architecture['components'])}):")
for c in architecture["components"]:
    print(f"  {c['name']:<20} {c['role']}")
    print(f"    Config: {c['config']}")

print(f"\nDeployment:")
for k, v in architecture["deployment"].items():
    print(f"  {k}: {v}")

print(f"\nSafety Checklist ({len(architecture['safety'])}):")
for i, s in enumerate(architecture["safety"], 1):
    print(f"  [{i}] {s}")

print(f"\nSandbox Hierarchy (weakest -> strongest):")
for i, level in enumerate([
    "Language restrictions (bypassable)",
    "Docker --cap-drop ALL (baseline)",
    "gVisor (user-space kernel)",
    "Firecracker microVMs (gold standard)",
], 1):
    print(f"  {i}. {level}")

print(f"\nOWASP LLM Top 10 (relevant):")
for risk in [
    "LLM01: Prompt Injection (tool results hijack agents)",
    "LLM05: Improper Output Handling (treat as untrusted)",
    "LLM06: Excessive Agency (over-privileged tools)",
]:
    print(f"  {risk}")